<a href="https://colab.research.google.com/github/sr00t3d/colab/blob/main/monitor_rdap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
%%writefile monitor_rdap.sh
#!/bin/bash
# monitor_rdap.sh - Verificação segura em lote com controle de taxa

if [ "$#" -eq 0 ]; then
  echo "❌ Erro: Nenhum domínio informado para consulta." >&2
  echo "Uso: $0 <dominio1.com.br> [dominio2.com.br ...]" >&2
  exit 1
fi

DOMAINS=("$@")
ALERT_DAYS="${ALERT_DAYS:-30}"

for domain in "${DOMAINS[@]}"; do
  # Executar requisição
  RESPONSE=$(curl -sf "https://rdap.registro.br/domain/$domain" || true)

  if [ -n "$RESPONSE" ]; then
    EXPIRY=$(echo "$RESPONSE" | jq -r '.events[]? | select(.eventAction=="expiration") | .eventDate' 2>/dev/null || true)

    if [ -n "$EXPIRY" ]; then
      EXPIRY_EPOCH=$(date -d "$EXPIRY" +%s)
      NOW_EPOCH=$(date +%s)
      DAYS_LEFT=$(( ($EXPIRY_EPOCH - $NOW_EPOCH) / 86400 ))

      if [ "$DAYS_LEFT" -le "$ALERT_DAYS" ]; then
        echo "⚠️  ALERTA: $domain expira em $DAYS_LEFT dias ($EXPIRY)"
      else
        echo "✅ $domain: OK ($DAYS_LEFT dias restantes)"
      fi
    fi
  else
    echo "❌ Erro ao consultar $domain"
  fi

  # Delay básico para respeitar rate limiting
  sleep 1
done

# Verificar cabeçalhos de controle de taxa em uma resposta
curl -sI "https://rdap.registro.br/domain/example.com.br" | grep -i "ratelimit\|retry-after" || true

Overwriting monitor_rdap.sh


Execution note: Since Registro.br returns the `nicbr-permission-denied: 1` blocking header for data centers, queries for actual domains will result in a direct connection error within the Colab VM; in this environment, they serve primarily for educational purposes regarding code structure.

In [16]:
# @title 🔍 Painel de Monitoramento RDAP (Registro.br)
# @markdown Insira os domínios separados por espaço:
dominios = "empresa.com.br marca.com.br produto.com.br"  # @param {type:"string"}
dias_alerta = 30  # @param {type:"integer"}

!ALERT_DAYS={dias_alerta} bash monitor_rdap.sh {dominios}

❌ Erro ao consultar empresa.com.br
❌ Erro ao consultar marca.com.br
❌ Erro ao consultar produto.com.br
